# 01 — Data collection & exploration

**Owners:** game-side tables: minhnguyen1210 · player-side tables: CaoThien1402

Spec sections 3.1 (data collection) and 3.2 (data exploration).

**Inputs:** TODO: files this notebook reads (`data/raw/<platform>/<table>.csv` or `data/processed/...`).

**Outputs:** TODO: files this notebook writes to `data/processed/`, and figures saved to `reports/figures/`.

In [1]:
import sys
from pathlib import Path

# Make the repo root importable (works from the repo root or from notebooks/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import RAW_DIR, PROCESSED_DIR, FIG_DIR, PLATFORMS, TABLES, RANDOM_STATE
from src.data_loading import load_table, load_all
from src.utils import report_rows

pd.set_option("display.max_columns", 50)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1 Data collection

This section documents what the data describes, where it comes from, under which license it is released and how it was
collected. Facts about the source come from the dataset's Kaggle page and from the author's open-source collection
project, GameStatsHub (both accessed on 21 November 2026). Statements about the content of the files were checked against
the raw data; the code that measures them is in section 2.

### 1.1 Subject

The dataset describes three major video game platforms, **Steam** (PC), **PlayStation** and **Xbox**, from two
complementary sides:

- **Games (supply side):** the game catalogue of each platform (title, developers, publishers, genres, supported
  languages, release date), the achievements or trophies defined for each game, and store prices recorded on several
  dates in five currencies (USD, EUR, GBP, JPY, RUB).
- **Players (demand side):** public player profiles, the games in each player's library and, for a subset of players,
  a time-stamped history of the achievements they unlocked. For Steam only, it also contains user reviews (text,
  "helpful" and "funny" votes, awards) and friend lists.

**Real-world context.** Achievements (Steam, Xbox) and trophies (PlayStation) are in-game milestones that the platform
records with a timestamp when a player reaches them. A player's unlock history is therefore a behavioural trace of which
games they play, when, and how far they progress, observed directly rather than self-reported in a survey. Store prices
recorded repeatedly show how publishers price and discount their games over time, including during Steam's seasonal
sales.

**Scale.** 21 CSV tables (six per platform plus three Steam-only tables) with about 56.4 million rows in total; the
largest table, `history`, holds 45.5 million unlock events (row counts in section 2.1). Each table is narrow (1 to 8
columns), but the tables share keys (`gameid`, `achievementid`, `playerid`) and together hold 41 distinct table columns,
so the analysis combines tables (section 2.3) rather than relying on a single wide table.

### 1.2 Source

| Item | Detail |
| --- | --- |
| Platform | Kaggle |
| Title | *Gaming Profiles 2025 (Steam, PlayStation, Xbox)*, subtitled "Comparison of User Behavior Across Different Gaming Platforms" |
| URL | <https://www.kaggle.com/datasets/artyomkruglov/gaming-profiles-2025-steam-playstation-xbox> |
| Author | Artyom Kruglov (Kaggle user `artyomkruglov`) |
| Collection code | GameStatsHub, <https://github.com/Smipe-a/gamestatshub>. The Kaggle description links to this repository and its README links back to the Kaggle page. |
| Publication date | 25 February 2025: version 1, "Initial release", still the only version on 21 November 2026 |
| Version used | Version 1, pinned in `scripts/download_data.sh` so that every run analyses the same files |
| Format and size | 21 CSV files, 4,034,016,221 bytes (about 4.0 GB); checked in the cell below |
| Original sources | [Steam Web API](https://steamcommunity.com/dev) and the [Steam Store](https://store.steampowered.com); [Exophase](https://www.exophase.com) (PlayStation and Xbox games, leaderboards and player histories); [TrueTrophies](https://www.truetrophies.com) and [TrueAchievements](https://www.trueachievements.com) (missing PlayStation and Xbox game details); [PSPrices](https://psprices.com) (PlayStation and Xbox prices) |

The Kaggle description advertises "~60GB" of data. This figure refers to the author's full PostgreSQL database, which
GameStatsHub distributes as a 3.8 GB SQL dump that requires at least 60 GB of free disk space to restore. The Kaggle
release is a 4.0 GB CSV
export of the database tables.

In [2]:
# Inventory of the raw files: all 21 tables of version 1 must be present.
# Kaggle reports 4,034,016,221 bytes for version 1 of the dataset.
files = [(platform, name, RAW_DIR / platform / f"{name}.csv")
         for platform in PLATFORMS for name in TABLES[platform]]
inventory = pd.DataFrame(
    [{"platform": platform, "table": name, "size_MB": path.stat().st_size / 1e6}
     for platform, name, path in files]
)
total_bytes = sum(path.stat().st_size for _, _, path in files)
print(f"{len(files)} files, {total_bytes:,} bytes in total")
inventory.pivot(index="table", columns="platform", values="size_MB").round(1)

21 files, 4,034,016,221 bytes in total


platform,playstation,steam,xbox
table,,,
achievements,70.1,144.8,29.0
friends,NaN,658.1,NaN
games,3.4,17.1,1.3
history,799.1,647.6,633.3
players,10.7,18.4,5.4
prices,2.3,183.9,0.8
private_steamids,NaN,4.3,NaN
purchased_games,74.4,91.7,86.5
reviews,NaN,551.6,NaN


### 1.3 License

The dataset is released under **CC0 1.0 Universal (Public Domain Dedication)** on Kaggle, and the GameStatsHub
repository carries the same license. Under CC0 the author waives copyright and related rights, so the data may be copied,
modified and redistributed, including for commercial purposes, without asking for permission. Attribution is not legally
required; we nevertheless cite the author and the repository, because academic practice and the course brief require
traceable sources.

Two restrictions still apply to how we use the data:

- **Rights in the original content.** CC0 covers the author's compilation. The underlying material (game descriptions,
  achievement texts, review texts) originates from Valve, Sony, Microsoft and the websites listed in section 1.2, whose
  terms of use govern the original content. We use it only for this non-commercial course project.
- **Personal data.** The tables contain real player identifiers (Steam IDs, PlayStation and Xbox nicknames) and
  user-written review text. The profiles were public when collected, but we still treat these fields as personal data:
  we report aggregate results only and never single out, name or try to re-identify an individual player.

### 1.4 Collection method

**How the data was collected.** The data was gathered by automated web scraping and public APIs. GameStatsHub runs a set
of Python scripts for each platform, orchestrated with Apache Airflow, and stores the results in a PostgreSQL database
(Figure 1); the Kaggle files are a CSV export of that database. The scripts in the repository show how each table was
filled:

| Tables | Source | Procedure |
| --- | --- | --- |
| Steam `players`, `friends` | Steam Web API (`GetPlayerSummaries`, `GetFriendList`) | Snowball sampling: starting from seed accounts taken from different sections and game categories of Steam, the crawler follows friend lists to new accounts. `country` is the optional location shown on the profile; `created` is the account creation time. |
| Steam `purchased_games`, `history` | Steam Web API (`GetOwnedGames`, `GetPlayerAchievements`) | Players are processed in random order. For each player, the owned games and the unlocked achievements (with timestamps) of games already in the database are stored. Private profiles get an empty library. |
| Steam `reviews`, `private_steamids` | Each player's public review pages | Reviews of games in the database are scraped, with the vote counts parsed from the review header. Players for whom no review can be collected are stored in `private_steamids`. |
| Steam `games`, `achievements` | Steam Web API (`GetAppList`, `GetSchemaForGame`) and the US Steam Store (`appdetails`) | Catalogue of all Steam apps with their store details and achievement lists. |
| PlayStation and Xbox `players` | Exophase leaderboards | All leaderboard pages are scraped; player id and nickname come from each profile page. PlayStation `country` is read from the flag shown on the leaderboard; Xbox players have no country. |
| PlayStation and Xbox `games`, `achievements` | Exophase game archive; TrueTrophies (PlayStation) and TrueAchievements (Xbox) | Games and their trophies (PlayStation `rarity` grade) or achievements (Xbox `points`). Missing developers, publishers, genres and release dates are filled from TrueTrophies or TrueAchievements by fuzzy title matching (Levenshtein distance). |
| PlayStation and Xbox `purchased_games`, `history` | Exophase API | Players are processed in random order; the games on each Exophase profile and their unlock timestamps are stored. |
| `prices` (all platforms) | Steam Store API; PSPrices for PlayStation and Xbox | On each collection day, the current price is recorded in five store regions (US, DE, GB, JP, RU). For Steam this is the store's `final` price, i.e. the price after any active discount. Console prices are found by searching PSPrices for the game title and platform and keeping the closest title match. |

![GameStatsHub data pipeline](../reports/figures/collect_gamestatshub_pipeline.png)

*Figure 1. GameStatsHub data pipeline. Source: [Smipe-a/gamestatshub](https://github.com/Smipe-a/gamestatshub),
`img/data_pipeline.png` (CC0).*

**Structure.** Each platform is stored as its own schema with the same core tables. Figure 2 shows the Steam schema. The
PlayStation and Xbox schemas contain the same six core tables without `reviews`, `friends` and `private_steamids`, and
differ only in a few columns: `games.platform` (PS4, PS5, PS3, PS Vita) and `achievements.rarity` (Bronze, Silver, Gold,
Platinum) on PlayStation, `achievements.points` on Xbox, and `players.nickname` on both consoles (plus `players.country`
on PlayStation) instead of Steam's `country` and `created`. The primary keys define what one row is: a game (`gameid`),
an achievement (`achievementid`), a player (`playerid`), one unlock (`playerid`, `achievementid`) and one price
observation (`gameid`, `date_acquired`).

![Steam schema of the GameStatsHub database](../reports/figures/collect_gamestatshub_steam_erd.png)

*Figure 2. Entity-relationship diagram of the Steam tables. Source:
[Smipe-a/gamestatshub](https://github.com/Smipe-a/gamestatshub), `img/steam_erd.png` (CC0).*

**Population.** The data is not a random sample of gamers.

| | Steam | PlayStation | Xbox |
| --- | --- | --- | --- |
| How players enter the data | snowball sampling over friend lists | Exophase leaderboard (ranked by achievement points) | Exophase leaderboard (ranked by achievement points) |
| Player profiles (`players`) | 424,683 | 356,600 | 274,450 |
| Players with an unlock history (`history`) | 4,838 | 5,000 | 4,984 |

**Period.** The GameStatsHub repository was created on 2 October 2024 and the last price snapshot is dated
25 February 2025, so the collection took place roughly between these two dates. The events it records are much older.
The author states a temporal coverage from 12 September 2008 to 12 January 2025; in the files, unlock timestamps run
from 12 September 2008 to 29 December 2024,
reviews from 15 October 2010 to 9 January 2025 and Steam account creation dates from 11 September 2003 to 7 January 2025.
Prices cover 45 dates, one every two days, for Steam (28 November 2024 to 24 February 2025), but only three dates for
PlayStation and two for Xbox (22 to 25 February 2025). Section 2.4 measures these ranges.

**Known limitations and biases.**

1. **Selection bias.** Console players come from leaderboards that rank players by achievement points, so they are
   dedicated achievement hunters. Steam players are reached through friend lists, which favours socially connected users
   with public profiles. Findings about behaviour describe these groups, not the average player.
2. **Small behavioural subsample.** Unlock histories exist for only about 5,000 players per platform, 1 to 2 % of the
   collected profiles.
3. **Public data only.** Private profiles leave gaps: 55,607 of 102,548 Steam libraries (54.2 %) are empty, and 41.9 %
   of Steam players have no `country` (section 2.1). These gaps are unlikely to be random.
4. **Parsing artefacts in `reviews`.** The vote counts are read with a pattern that only matches "N people found this
   review helpful". Steam writes a single vote as "1 person", so one vote is stored as 0: no review has `helpful = 1`,
   whereas 98,268 have `helpful = 2`. Counts written with a thousands separator keep only their last three digits
   ("1,234" becomes 234 and "1,000" becomes 0), so `helpful` never exceeds 999. `funny` is parsed in the same way.
   In addition, `posted` holds the date of the last edit when a review was edited.
5. **Misleading table names.** `private_steamids` lists every Steam player for whom no review could be collected,
   whether the profile is private or the player never wrote a review: the 196,698 reviewers and the 227,963 players in
   `private_steamids` together make up all but 22 of the 424,683 Steam players. PlayStation and Xbox `purchased_games`
   hold the games shown on an Exophase profile, which are games played, not necessarily purchased.
6. **Matching by title.** Console prices (PSPrices) and missing console game details (TrueTrophies, TrueAchievements)
   are matched to games by title, which can attach the data to the wrong game or edition.
7. **Incomplete export.** In the author's schema, `prices` and `achievements` refer to `games` through a foreign key, yet
   7,570 PlayStation and 830 Xbox game ids that appear in `achievements` and `prices` have no row in `games.csv`; on
   Steam, 217 priced games and 77 games with achievements are missing as well (section 2.3).
8. **Snapshots and time zones.** Profiles, libraries and friend lists are a single snapshot taken between late 2024 and
   early 2025, not the state at the time of past events. Unlock times were converted from UNIX timestamps with the local
   time zone of the collecting machine, which is not documented, so clock times may be shifted by a constant offset.
9. **No updates.** The dataset is described as "Regularly Updated", but version 1 is the only release, so the data ends
   in early 2025.

### 1.5 Motivation

Video games are a topic both of us enjoy, and one of us regularly follows news about the major gaming platforms. That
coverage made us curious about what lies behind the headlines: how the market for games actually works, and how other
players behave once a game is in their library. This dataset lets us study both questions with real data instead of
anecdotes.

Four features make it a good fit for our project:

- **Three platforms, one table structure.** Steam, PlayStation and Xbox share the same core tables, so the platforms
  can be compared directly rather than through separate, incompatible sources.
- **Observed behaviour over 16 years.** 45.5 million time-stamped achievement unlocks (2008 to 2024) record what players
  actually did, not what they say they did in a survey.
- **Prices during real sales.** Steam prices recorded every two days from November 2024 to February 2025 cover two major
  sales, at whose peaks about 60 % of the games priced throughout the period were discounted.
- **Real, imperfect data.** Four gigabytes of CSV files in 21 linked tables, with missing values and collection
  artefacts (section 1.4), call for the loading, cleaning and joining skills this course teaches. The dataset is also
  public, released under CC0, traceable to its collection code and still little analysed: Kaggle lists only nine public
  notebooks for it (November 2026).

Our findings can inform several stakeholders: **players** deciding when and on which platform to buy a game;
**independent developers** setting prices and designing achievements that keep players engaged; **platform operators**
planning sales and engagement features; and **researchers** studying player behaviour across gaming ecosystems.

## 2 Data exploration

Go deep on the columns relevant to our questions; summarise the rest briefly instead of one chart per column.

Each subsection has one code cell per table group, so the two members can work in parallel:
**game side** (`games`, `achievements`, `prices`) and **player side** (`players`, `history`, `purchased_games`,
`reviews`, `friends`, `private_steamids`).

### 2.1 Loading strategy

**Player-side tables** (`players`, `history`, `purchased_games`, `reviews`, `friends`, `private_steamids`; owner: CaoThien1402).
The game-side tables (`games`, `achievements`, `prices`) are covered by minhnguyen1210 in the first code cell of each subsection.

All player-side tables fit in memory (the largest, PlayStation `history`, needs 1.6 GB), so nothing is read in chunks
for now. The settings live in `PLAYER_READ_ARGS` in `src/data_loading.py` and are used through `load_player_table`.

| Decision | Tables | Why |
| --- | --- | --- |
| `int64` for Steam ids, `int32` for PlayStation and Xbox ids | all | Steam ids are about 7.6e16 and overflow `int32`; the other platforms use ids below 2^31 |
| `parse_dates` | `history.date_acquired`, `players.created`, `reviews.posted` | dates are read as text otherwise, so time arithmetic and resampling need a conversion later |
| `country` as `category` | Steam, PlayStation `players` | few distinct values repeated over hundreds of thousands of rows: the Steam table drops from 56 MB to 8 MB |
| `achievementid` as `string` | `history` | the id is text (`403640_ACH_1`); `history` uses 42-47% less memory than with the defaults (measured on 200,000 rows) |
| `usecols` without `review` | `reviews` | the review text is about 94% of the memory (715 MB with the text, 43 MB without it); Q4 reads the text only if it builds text features |
| `keep_default_na=False, na_values=[""]` | `players` | only an empty cell is missing; with the default setting pandas also turns the nickname `null` (one Xbox player) into `NaN` |
| one table at a time, then `del` | `history`, `reviews`, `friends` | keeps memory below about 2 GB; reading `history` takes 20-33 s per platform |

**Columns that hold a list as text.** `library` (`purchased_games`) and `friends` (`friends`) store lists such as
`"[60, 1670]"`. Parsing about 65 million ids with `ast.literal_eval` would be slow, so the tables are kept as text and
`count_list_items` counts the items (number of commas plus one). `parse_id_list` turns a single cell into a Python list
when the ids themselves are needed (for example for `explode`).

**Empty list cells are real missing data, not an empty list.** In the raw file, 55,607 of 102,548 Steam libraries
(54.2%) and 85,222 of 424,683 friend lists (20.1%) are empty cells; the text `"[]"` never appears. PlayStation and Xbox
libraries have no empty cell. Most likely these are private profiles, but the dataset does not say so; we treat them as
unknown, not as "owns no game". Any statement about library size or friends is therefore limited to the players who
made them public.

**Other things checked while loading**
- `reviews` contains text cells that span several lines. The default C parser of pandas reads them correctly (1,204,534
  rows, the same number as the author's description), so we do not use the `pyarrow` engine for this table.
- Steam `players.country` has 177,868 empty cells (41.9%); PlayStation `players` has a `country` column, Xbox has none.
  See section 2.7.
- `private_steamids` has no duplicated id (227,963 rows).

The two tables below show the result for every player-side table: size on disk, rows, memory after loading, saving
from the dtype settings on a 200,000-row sample, and read time. `reviews` is shown with its text (see above).

In [3]:
# Game side: games, achievements, prices (owner: TODO)

In [4]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: CaoThien1402)
import gc
import time

from src.data_loading import PLAYER_SIDE_TABLES, count_list_items, load_player_table, table_path

SAMPLE_ROWS = 200_000  # sample used to compare memory with and without the settings in PLAYER_READ_ARGS

records = []
for platform, name in PLAYER_SIDE_TABLES:
    # memory saved by dtype / parse_dates: same first rows, default settings vs PLAYER_READ_ARGS
    default_mb = load_table(platform, name, nrows=SAMPLE_ROWS).memory_usage(deep=True).sum() / 1e6
    tuned_mb = load_player_table(platform, name, nrows=SAMPLE_ROWS).memory_usage(deep=True).sum() / 1e6

    start = time.perf_counter()
    df = load_player_table(platform, name)  # the full table, one at a time
    seconds = time.perf_counter() - start
    records.append({
        "platform": platform, "table": name,
        "file_MB": round(table_path(platform, name).stat().st_size / 1e6),
        "rows": len(df), "columns": df.shape[1],
        "memory_MB": round(df.memory_usage(deep=True).sum() / 1e6),
        "sample_saving_%": round(100 * (1 - tuned_mb / default_mb)),
        "seconds": round(seconds, 1),
    })
    del df
    gc.collect()

loading_summary = pd.DataFrame(records)
loading_summary

,platform,table,file_MB,rows,columns,memory_MB,sample_saving_%,seconds
0,steam,players,18,424683,3,8,86,0.3
1,playstation,players,11,356600,3,26,49,0.1
2,xbox,players,5,274450,2,20,5,0.1
3,steam,history,648,10693879,3,1001,42,8.3
4,playstation,history,799,19510083,3,1571,47,15.1
5,xbox,history,633,15275900,3,1239,47,16.2
6,steam,purchased_games,92,102548,2,95,0,0.9
7,playstation,purchased_games,74,46582,2,77,0,0.4
8,xbox,purchased_games,86,46466,2,89,0,0.5
9,steam,reviews,552,1204534,8,715,13,7.6


In [5]:
# Columns that store a list as text: how many cells are empty, and how many ids do they hold?
list_columns = [("steam", "purchased_games", "library"), ("playstation", "purchased_games", "library"),
                ("xbox", "purchased_games", "library"), ("steam", "friends", "friends")]
rows = []
for platform, name, column in list_columns:
    df = load_player_table(platform, name)
    items = count_list_items(df[column])
    rows.append({"platform": platform, "table": name, "rows": len(df),
                 "empty_cells": int(df[column].isna().sum()),
                 "empty_%": round(100 * df[column].isna().mean(), 1),
                 "ids_in_lists": int(items.sum()), "max_per_row": int(items.max())})
    del df
pd.DataFrame(rows)

,platform,table,rows,empty_cells,empty_%,ids_in_lists,max_per_row
0,steam,purchased_games,102548,55607,54.2,11258715,32463
1,playstation,purchased_games,46582,0,0.0,10893552,13540
2,xbox,purchased_games,46466,0,0.0,12667877,9018
3,steam,friends,424683,85222,20.1,30903334,2000


### 2.2 Overview & data dictionary

TODO, per table used:
- number of rows and columns; what one row represents;
- meaning and data type of each column; columns to drop or convert;
- duplicated or empty rows, and whether to remove them (justify).

| Table | Column | Type | Meaning | Action (keep / convert / drop) |
| --- | --- | --- | --- | --- |
| TODO | TODO | TODO | TODO | TODO |

In [6]:
# Game side: games, achievements, prices (owner: TODO)

In [7]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.3 Keys & joins between tables

TODO: for every join the questions will need (e.g. `history` → `achievements` → `games`, `prices` → `games`,
`reviews` → `players`): rows before and after, and the number of keys without a match on each side.

In [8]:
# Game side: games, achievements, prices (owner: TODO)

In [9]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.4 Time coverage

TODO: for every date column, per platform: first and last date, frequency, gaps; suspicious dates (placeholders,
spikes); how the end of data collection limits recent periods.

In [10]:
# Game side: games, achievements, prices (owner: TODO)

In [11]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.5 Numerical columns

TODO: distribution shape, centre and spread; plausible min/max; outliers (IQR, z-score) genuine or errors;
impossible or placeholder values.

In [12]:
# Game side: games, achievements, prices (owner: TODO)

In [13]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.6 Categorical columns

TODO: number of distinct values and most frequent ones; balance; inconsistent labels and typos;
rare categories to group.

In [14]:
# Game side: games, achievements, prices (owner: TODO)

In [15]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.7 Missing data

TODO: count and percentage per column; whether missingness follows a pattern; why values are missing;
how each column will be handled (drop, impute, separate category).

In [16]:
# Game side: games, achievements, prices (owner: TODO)

In [17]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

### 2.8 Relationships

TODO: correlations between numerical variables; cross-tabulations and grouped statistics for important
combinations; surprising relationships.

In [18]:
# Game side: games, achievements, prices (owner: TODO)

In [19]:
# Player side: players, history, purchased_games, reviews, friends, private_steamids (owner: TODO)

## 3 Exploration summary

### 3.1 Key observations (3–5)

TODO

### 3.2 Serious data quality issues

TODO

### 3.3 Preprocessing plan

TODO: the cleaning steps shared by several questions, and the files they produce.

| Step | Tables | Rule | Output in `data/processed/` | Used by |
| --- | --- | --- | --- | --- |
| TODO | TODO | TODO | TODO | TODO |

### 3.4 Patterns that could lead to questions

TODO